# Source E: Medi-Cal FFS provider list

This notebook consolidates the statewide Medi-Cal FFS provider list down to the Inland Empire (San Bernardino + Riverside counties), keeps the needed provider and geographic attributes, cleans the data, and removes duplicate rows when the same provider appears multiple times for the same address.

Run the code cells from top to bottom with Shift+Enter.

## 1. Find the data and import pandas

In [ ]:
from pathlib import Path
import pandas as pd

relative_partition_dir = Path(
    "data/provider-side/E-mediCal-fee-for-service-enrolled-providers/"
    "zipped_bc_too_big_for_github"
)

repo_root = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / relative_partition_dir).is_dir()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the Source E data from the notebook's current folder: "
        f"{Path.cwd()}. Start Jupyter with the repository folder as its root."
    )

partition_dir = repo_root / relative_partition_dir
print("Source E folder:", partition_dir)


## 2. Load the two CSV parts

In [ ]:
csv_files = [
    partition_dir / "medi-cal-ffs-provider-listing_10_26_26_PART 1 first 200k line.csv",
    partition_dir / "medi-cal-ffs-provider-listing_10_26_26_PART 2 next 180k line.csv",
]

providers_df = pd.concat(
    [
        pd.read_csv(
            path,
            encoding="cp1252",
            low_memory=False,
            keep_default_na=False,
            na_values=[""],
        )
        for path in csv_files
    ],
    ignore_index=True,
)

print("Loaded both CSV parts into providers_df.")
print("Rows loaded:", len(providers_df))


## 3. Clean, subset to San Bernardino + Riverside, and deduplicate

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().replace("\u00a0", " ").replace("  ", " ")

providers_df = providers_df.copy()

# Normalize object/string fields to clean leading/trailing whitespace and weird spacing
for col in providers_df.columns:
    if providers_df[col].dtype == "object":
        providers_df[col] = providers_df[col].map(normalize_text)

# Keep the most useful geographic fields and provider classifications
county_cols = ["County", "CountyName"]
for col in county_cols:
    providers_df[col] = providers_df[col].fillna("").astype(str).str.strip()

providers_df["County_norm"] = providers_df["County"].fillna("").str.upper()
providers_df["CountyName_norm"] = providers_df["CountyName"].fillna("").str.upper()
providers_df["State_norm"] = providers_df["State"].fillna("").str.upper()

county_targets = {"SAN BERNARDINO", "RIVERSIDE"}
county_name_targets = {"SAN BERNARDINO COUNTY", "RIVERSIDE COUNTY"}

inland_empire_df = providers_df[
    (providers_df["County_norm"].isin(county_targets))
    | (providers_df["CountyName_norm"].isin(county_name_targets))
].copy()

print("Rows in statewide dataset:", len(providers_df))
print("Rows after Inland Empire county filter:", len(inland_empire_df))
print("County counts after filter:")
print(inland_empire_df["County"].value_counts(dropna=False))

# Create a stable provider key. If NPI is missing, fall back to Provider_Number.
inland_empire_df["provider_key"] = (
    inland_empire_df["NPI"].fillna(inland_empire_df["Provider_Number"])
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

# Create an address key so the same provider with different service locations is preserved
inland_empire_df["address_key"] = (
    inland_empire_df["Address"].fillna("").astype(str).str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
    + " | " + inland_empire_df["Address2"].fillna("").astype(str).str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
    + " | " + inland_empire_df["City"].fillna("").astype(str).str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
    + " | " + inland_empire_df["State"].fillna("").astype(str).str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
    + " | " + inland_empire_df["ZIP"].fillna("").astype(str).str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
).str.replace(r"\s+\|\s+", " | ", regex=True)

# Deduplicate only when provider + exact address are the same. This retains true multi-site providers.
inland_empire_df = inland_empire_df.drop_duplicates(
    subset=["provider_key", "address_key"],
    keep="first",
).copy()

print("Rows after provider+address deduplication:", len(inland_empire_df))

keep_cols = [
    "Provider_Number",
    "NPI",
    "NPI_Type",
    "Owner_Number",
    "Service_Location_Number",
    "Legal_Name",
    "Enroll_Status_Eff_DT",
    "Provider_Taxonomy",
    "NEMT_NMT_Provider_Type",
    "ANC_Provider_Type",
    "FI_Provider_Type_CD",
    "FI_Provider_Type",
    "Provider_License",
    "FI_Provider_Specialty_CD",
    "FI_Provider_Specialty",
    "Out_of_State_Indicator",
    "In_Out_State",
    "Address_Attention",
    "Address",
    "Address2",
    "City",
    "State",
    "ZIP",
    "ZIP_4",
    "DHCS_County_CD",
    "FIPS_County_CD",
    "County",
    "CountyName",
    "Phone_Number",
    "Medicaid_Patients",
    "CHIP_Patients",
    "Telehealth_Services",
    "Provider_Website",
    "Threshold_Languages",
    "Other_Languages",
    "Acc_Exam_Room",
    "Acc_Exterior_Building",
    "Acc_Interior_Building",
    "Acc_Parking",
    "Acc_Restroom",
    "Acc_Medical_Equipment",
    "Acc_Patient_Areas",
    "Acc_Patient_Diagnostic",
    "Provider_Source",
    "Billing_Provider",
    "Latitude",
    "Longitude",
]

inland_empire_clean = inland_empire_df[keep_cols].copy().reset_index(drop=True)
inland_empire_clean["County"] = inland_empire_clean["County"].replace({"": "Unknown"})
inland_empire_clean["CountyName"] = inland_empire_clean["CountyName"].replace({"": "Unknown"})
inland_empire_clean["State"] = inland_empire_clean["State"].replace({"": "CA"})

print("Final row count:", len(inland_empire_clean))
print("Top provider types:")
print(inland_empire_clean["FI_Provider_Type"].value_counts(dropna=False).head(10))
print("County breakdown:")
print(inland_empire_clean["County"].value_counts(dropna=False))

output_path = partition_dir / "san_bernardino_riverside_ffs_providers_clean.csv"
inland_empire_clean.to_csv(output_path, index=False)
print("Saved cleaned file to:", output_path)


## 4. Preview the clean Inland Empire table

In [ ]:
inland_empire_clean.head()
